# The master tables — decodability BEFORE editability

Reads the `scores.json` files `master_eval.ipynb` writes (scanning `runs/**`, excluding
`runs/archive/` and **any path with a `_`-prefixed component — a whole topic like
`runs/_pipeline_smoke/` or a single run like `runs/<topic>/_<run>/`**) and renders them as
**heatmap figures** — publication-ready, copy-pasteable, and displayed inline. Nothing is
cached to disk: each run's `scores.json` is the single source of truth and cell [1]
re-derives these in under a second. Cell [1] prints which runs it excluded.

Renaming a run with a leading `_` is therefore how a run leaves the tables: nothing moves,
nothing is deleted, `master_eval` keeps scoring it, and only this notebook ignores it.

**Order is deliberate: decodability first.** An editability number is only interpretable
once the probes demonstrably read the state — an editor writing through a probe that
decodes nothing produces noise, not a negative result.

**Row order is the same in every table:** environment first (Othello, then discworld),
then architecture (Transformer-L, then Recurrent-L), then instance (`dw-pn04`,
`dw-noiseless`, `dw-8ray`, `dw-blink`). A run's rows are labelled once, as a block, and its
probe targets are told apart by the row label alone.

* **Table 1** — decodability (Probe Skill, the cross-environment axis: 1 = perfect,
  0 = trivial baseline; identical to R² on regression). Every probe is held out **by
  sequence** in both environments.
* **Tables 1b / 1c** — discworld decodability **by component**, for the linear probe
  (1b) and the MLP (1c). The aggregate is variance-weighted ~1000:1 toward position, so
  velocity is only visible here.
* **Tables 1d / 1e** — the same, **minus the random-init floor** per component (the
  same architecture, untrained, same instance and basis): what training added to each
  state variable, possibly negative. Read from the cached baseline probes.
* **Table 2** — editability. Panel (a) is the Edit Index with each run's **unedited**
  floor as its first column — that is where the −1 end actually sits, so every editor
  must be read against it. Panel (b) is the guard, `RMSE(edited, GT)/RMSE(unsteered, GT)`
  at the edit step: **> 1 means the edit degraded the model rather than steering it**,
  whatever the index says.
* **Table 2b** — the arm behind every Table 2 cell.
* **Tables 3a / 3b / 3c…** — the decodability baselines (observation and random-init
  floors) against the trained rows, one table per discworld probe target (cartesian 3a,
  frustum 3b, Othello in both; then one per extra target, discworld only).

**One row per (run, probe target)** — a probe target is a different row, never a new
column. For a discworld run that is one row per **basis** of the regression target
(cartesian, frustum) plus one per **extra target** it carries. The one extra target so
far is **`grid-16x8`** (2026-09-09): the discworld state as 16 × 8 = 128 frustum-uniform
cells × {empty, object 0, object 1} — Othello's shape of target, 3-way classification per
cell, on the same trained model. Its probes exist for `noise_ablation/L-dw-noiseless-20m`
only (fitted 2026-09-08, `experiments/grid_target_control`, canonicalised 2026-09-09), so
that run has three rows and every other discworld run two. Read that row like an Othello
row: Probe Skill is 1 − err/majority-err, the edit is a categorical MOVE (old cell → empty,
new cell → the object; teleports that stay in one cell are excluded from its 192-case
bench), and **ND is reported** there, as on Othello — a categorical edit is one fixed
change per case, which is exactly the condition ND needs and the regression target lacks.

**Discworld fits one probe set per basis, on the full state.** Editability is then swept
twice through it: driving the position read-outs only (`pos`) and driving the whole state
(`all`); the better arm is reported, and Table 2b names which one won. The retired
pos-only probes are not missing from this — for the **linear** probe the position rows of
a full-state least-squares fit are bit-identical to a position-only fit (multi-output
least squares decomposes per output dimension), so `pos` reproduces them exactly. The MLP
does not decompose, so for GS the two dim sets are genuinely different edits.

**† — the frame-set Edit Index (2026-09-05).** A discworld run marked † was trained on
frames as *tokens* (the Othello architecture over the instance's frame vocabulary) and emits
a distribution over next frames, so its Edit Index is Othello's legal-set construction
applied to the frames the two worlds render at the edit frame — the same axis and the same
step-0 reading as the ray-zone index of the other discworld rows, but not the same formula.
Its guard is Othello's `move_fidelity_ratio`. Probe Skill, the probes and the editors are
the ordinary discworld ones (regression probes on the residual stream; PI and GS).

In [ ]:
# [1] Collect every scores.json (runs/**; archive/ and _-prefixed paths excluded) -> tidy frames.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))        # Fig 1/2 import the canonical palette + figure code
EDITORS = ("PI", "ND", "GS")
COMPONENTS = ("o1·x", "o1·y", "o2·x", "o2·y", "o1·vx", "o1·vy", "o2·vx", "o2·vy")
SHORT = {"transformer_l": "L", "transformer_s": "S", "recurrent_l": "R",
         "transformer_l_tokens": "L", "transformer_s_tokens": "S"}


# ⛔ WHICH RUNS COUNT — one rule, used by every scan in this notebook.
# A path component starting with "_" excludes the run AT ANY DEPTH (2026-09-08): a whole
# quarantined topic (runs/_pipeline_smoke/, runs/_architecture_gate/) and a single run
# pulled out of the record (runs/<topic>/_<run>/) are spelled the same way, so a run leaves
# the tables by being RENAMED — no move, nothing deleted from runs/, and its scores.json
# stays on disk for master_eval to keep current. runs/archive/ is excluded by name as always.
def counted(rel: Path) -> bool:
    """Does runs/<rel> belong in the tables? `rel` = a scores.json path relative to runs/."""
    return rel.parts[0] != "archive" and not any(p.startswith("_") for p in rel.parts)


def gap_at_best(T, fam, key_):
    """The in-sample − held-out gap at the point the skill is quoted from (Table 3 puts the
    trained model on the same axis as its two floors)."""
    ps = {r["point"]: r for r in T["probe_sanity"]["rows"]}
    bp = int(np.argmax(T[f"probe_skill_{fam}"]))
    return ps.get(bp, {}).get(f"insample_gap_{fam}", np.nan)


rows, perdim_rows, skipped = [], [], []
for sp in sorted((REPO / "runs").rglob("scores.json")):
    rel = sp.relative_to(REPO / "runs")
    if not counted(rel):
        skipped.append(str(rel.parent))
        continue
    s = json.loads(sp.read_text())
    s_env = s["env"]
    base = {"topic": rel.parts[0], "run": sp.parent.name, "env": s_env,
            "instance": s["instance"], "arch": s["arch"], "val": s["val_loss"]}

    def editor_cells(row, best, key, env=s_env, kind="classification"):
        for ed in EDITORS:
            b = best.get(ed)
            # ⛔ ND is not applicable on discworld's REGRESSION target: one fixed direction
            # with a swept scalar cannot serve 192 teleports of differing magnitude and
            # direction. Computed and kept in scores.json, deliberately blank here
            # (2026-09-01). On a CATEGORICAL target (Othello; discworld's grid and appearance
            # targets, 2026-09-09) the edit is one categorical change per case and ND is
            # reported like any editor; on Othello's signed REGRESSION target the flip is a
            # constant-magnitude sign change, so ND is sound and reported there too.
            if ed == "ND" and env == "discworld" and kind == "regression":
                b = None
            row[f"{ed} EI"] = b[key] if b else np.nan
            row[f"{ed} fid"] = b["fidelity_ratio"] if b else np.nan
            # The arm behind the cell. On discworld it NAMES THE WINNING DIM SET —
            # each editor is swept driving position only and driving the whole state,
            # and the better arm is the reported one, so which won is part of the record.
            row[f"{ed} arm"] = ("—" if not b else
                                f"{b['dims']}·pt{b['point']}·α{b['alpha']:g}"
                                if "dims" in b else f"pt{b['point']}·α{b['alpha']:g}")

    if s_env == "discworld":
        if "bases" not in s:
            raise RuntimeError(f"{sp} predates the per-basis schema — rerun master_eval")
        # A PROBE TARGET is a different ROW — never a column that exists for only some
        # runs. `bases` holds one block per target: a basis of the regression target
        # (keyed by the basis) or an extra target such as the grid (keyed by its name);
        # each block names its `kind` (blocks written before 2026-09-09 are regression).
        for key, T in s["bases"].items():
            kind = T.get("kind", "regression")
            row = {**base, "basis": key, "kind": kind}
            row["skill_LIN"] = max(T["probe_skill_linear"])
            row["skill_MLP"] = max(T["probe_skill_mlp"])
            row["tripwire"] = T["probe_sanity"]["n_violations"]
            row["unedited"] = T["unedited"]["edit_index"]
            for fam, key_ in (("linear", "LIN"), ("mlp", "MLP")):
                row[f"gap_{key_}"] = gap_at_best(T, fam, key_)
            editor_cells(row, T["best"], "edit_index", kind=kind)
            rows.append(row)
            if kind != "regression":
                continue                    # no per-component skill on a categorical target
            # Per-component skill. The probe target is the FULL state (the pos-only set
            # was retired 2026-09-01), so velocity is always present here.
            bp = int(np.argmax(T["probe_skill_linear"]))
            for name, pdim in (("LIN", T["probe_perdim_linear"][bp]),
                               ("MLP", T["probe_perdim_mlp"][bp])):
                perdim_rows.append({"run": row["run"], "topic": row["topic"], "env": s_env,
                                    "arch": s["arch"], "instance": s["instance"],
                                    "basis": key, "probe": name, "point": bp,
                                    **dict(zip(COMPONENTS, pdim[:len(COMPONENTS)]))})
    else:
        # the canonical categorical block (top level, "mine/theirs") …
        row = {**base, "basis": "mine/theirs", "kind": "classification"}
        sk = s["probe_skill"]
        row["skill_LIN"] = max(sk.get("mine|linear|sequence", [np.nan]))
        row["skill_MLP"] = max(sk.get("mine|mlp|sequence", [np.nan]))
        row["tripwire"] = 0
        row["unedited"] = s["unedited"]["edit_index_union"]
        row["legal_mass"] = s["gates"]["legal_mass"]
        row["ce_excess"] = s["gates"]["ce"] - s["gates"]["bayes_ce"]
        for fam, key_ in (("linear", "LIN"), ("mlp", "MLP")):
            sub = [x for x in s["probe_stats"] if x["target"] == "mine"
                   and x["split"] == "sequence" and x["family"] == fam]
            b = min(sub, key=lambda x: x["error_rate"]) if sub else None
            row[f"gap_{key_}"] = ((b["error_rate"] - b["error_rate_insample"])
                                  / b["majority_class_error_rate"]) if b else np.nan
        editor_cells(row, s["best"], "edit_index_union")
        rows.append(row)
        # … and every extra-target block (2026-09-09: "mine_signed", the signed mine/theirs
        # REGRESSION target), the shared block shape — one more row per target
        for key, T in s.get("bases", {}).items():
            kind = T.get("kind", "regression")
            row = {**base, "basis": key, "kind": kind}
            row["skill_LIN"] = max(T["probe_skill_linear"])
            row["skill_MLP"] = max(T["probe_skill_mlp"])
            row["tripwire"] = T["probe_sanity"]["n_violations"]
            row["unedited"] = T["unedited"]["edit_index_union"]
            for fam, key_ in (("linear", "LIN"), ("mlp", "MLP")):
                row[f"gap_{key_}"] = gap_at_best(T, fam, key_)
            editor_cells(row, T["best"], "edit_index_union", kind=kind)
            rows.append(row)

# THE ROW ORDER, used by every table: environment first (Othello — the canonical instance,
# then the no-flip variant, the adjacency variant, then adjacency WITH recolouring
# (2026-09-09) — then discworld), then architecture (transformers before the
# recurrent model), then instance (the canonical noisy instance, the noiseless one, then
# the 8-ray radius-1.0 instance, then the blink instance — noiseless with blackouts,
# 2026-09-07), then run, probe target — a run's canonical targets (mine/theirs; cartesian,
# frustum) before its extra targets, the categorical ones from COARSE to FINE (cells:
# appearance-lat 15 · grid-4x2 8 · appearance 30 · grid-6x5 30 · grid-10x3 30 · grid-8x4 32
# · appearance-d2 60 · appearance-d3 90 · grid-16x8 128 · grid-32x16 512 · grid-64x32 2048),
# so the resolution sweep reads top-to-bottom within a run's block.
INSTANCE_ORDER = ("oth-uniform", "oth-noflip", "oth-adjacent", "oth-adjacent-flip",
                  "dw-pn04", "dw-noiseless", "dw-8ray", "dw-blink")
BASIS_ORDER = ("mine/theirs", "mine_signed", "cartesian", "frustum",
               "appearance-lat", "grid-4x2", "appearance", "grid-6x5", "grid-10x3", "grid-8x4",
               "appearance-d2", "appearance-d3", "grid-16x8", "grid-32x16", "grid-64x32")
def order_key(env, arch, inst):
    return (0 if env == "othello" else 1,
            0 if str(arch).startswith("transformer") else 1,
            INSTANCE_ORDER.index(inst) if inst in INSTANCE_ORDER else 9)
def ordered(df):
    if not len(df):
        return df
    k = [order_key(r.env, r.arch, r.instance) for r in df.itertuples()]
    kb = [BASIS_ORDER.index(b) if b in BASIS_ORDER else 99 for b in df["basis"]]
    return (df.assign(_o=k, _b=kb).sort_values(["_o", "run", "_b", "basis"])
            .drop(columns=["_o", "_b"]).reset_index(drop=True))

DF_ALL = ordered(pd.DataFrame(rows))
PERDIM_ALL = ordered(pd.DataFrame(perdim_rows))
# The training-curve runs (a canonical run's OWN checkpoints laid out as run dirs) are a
# view of one run over time, not new canonical runs: they get Fig 1 at the end and stay
# out of Tables 1-3, which 16 near-duplicate rows would otherwise bury.
_main = lambda d: d[d["topic"] != "training_curve"].reset_index(drop=True) if len(d) else d
DF, PERDIM = _main(DF_ALL), _main(PERDIM_ALL)
# The EXTRA probe targets (the categorical partitions, the snapped regression targets, Othello's
# signed target) leave the main tables for their own Table 2c (2026-09-10): Tables 1-3 keep the
# CANONICAL bases only — two bases per discworld run, mine/theirs per Othello run.
CANONICAL_BASES = ("cartesian", "frustum", "mine/theirs")
_extra = lambda d: ~d["basis"].isin(CANONICAL_BASES)
DF_X = DF[_extra(DF)].reset_index(drop=True) if len(DF) else DF     # Table 2c, Fig 3, the 3c… floors
DF = DF[~_extra(DF)].reset_index(drop=True) if len(DF) else DF
PERDIM = PERDIM[~_extra(PERDIM)].reset_index(drop=True) if len(PERDIM) else PERDIM
KIND_X = dict(zip(DF_X["basis"], DF_X["kind"])) if len(DF_X) else {}
# the extra probe targets present, per environment — each gets its own Table 3
EXTRA_TARGETS = sorted(set(DF_X[DF_X["env"] == "discworld"]["basis"]),
                       key=lambda b: BASIS_ORDER.index(b) if b in BASIS_ORDER else 99)
EXTRA_TARGETS_OTH = sorted(set(DF_X[DF_X["env"] == "othello"]["basis"]))
print(f"{len(DF)} canonical (run, basis) rows + {len(DF_X)} extra-target rows (Table 2c) · "
      f"{DF.env.value_counts().to_dict()}"
      + (f" · extra targets {EXTRA_TARGETS}" if EXTRA_TARGETS else "")
      + (f" · Othello extra targets {EXTRA_TARGETS_OTH}" if EXTRA_TARGETS_OTH else "")
      + (f" · +{len(DF_ALL) - len(DF)} training-curve rows (Fig 1)" if len(DF_ALL) > len(DF) else ""))
# Named, not silent: excluding a run is a deliberate act and the reader should see which.
print(f"excluded ({len(skipped)}): " + (", ".join(skipped) if skipped else "none"))

sns.set_theme(style="white", font_scale=0.95)
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})     # crisp inline tables
RULE = "#172239"

def heat(ax, data, xt, yt, *, fmt, cmap, norm=None, vmin=None, vmax=None, cbar_label,
         title):
    """One heatmap cell block — every table here is built through this, so the tables
    cannot drift apart in colormap, annotation format, or scaling. Rows TOUCH (no white
    gap between the bases of one run — the only horizontal rules are the block rules
    group_rows draws); columns keep a thin white separator."""
    sns.heatmap(data, annot=True, fmt=fmt, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax,
                xticklabels=xt, yticklabels=yt, linewidths=0,
                cbar_kws=dict(label=cbar_label, shrink=0.75, pad=0.02), ax=ax,
                annot_kws=dict(fontsize=9))
    for j in range(1, np.shape(data)[1]):
        ax.axvline(j, color="white", lw=1.2)
    ax.set_title(title, fontsize=10.5, loc="left", pad=8)
    ax.tick_params(labelsize=9, length=0)
    plt.setp(ax.get_yticklabels(), rotation=0)
    plt.setp(ax.get_xticklabels(), rotation=0)

def groups_of(df, key):
    """Consecutive runs of `key` in row order -> [(label, n_rows)], for group_rows."""
    out = []
    for v in df[key]:
        if out and out[-1][0] == v:
            out[-1][1] += 1
        else:
            out.append([v, 1])
    return out

def block_rules(ax, ys, texts=()):
    """Black rules between row blocks, at data-y positions `ys`. Given the group-label
    texts, each rule is extended LEFT — through the tick labels — to the labels' left
    edge, so a block boundary is unmistakable wherever the eye is."""
    x0 = 0.0
    if texts:
        fig = ax.figure
        fig.canvas.draw()                              # text extents need a render
        r, inv = fig.canvas.get_renderer(), ax.get_yaxis_transform().inverted()
        x0 = min(inv.transform((t.get_window_extent(r).x0, 0))[0] for t in texts) - 0.01
    for y in ys:
        ax.plot([x0, 1], [y, y], transform=ax.get_yaxis_transform(), clip_on=False,
                color=RULE, lw=1.6, solid_capstyle="butt")

def group_rows(ax, groups, x=-0.36):
    """Two-level row labels: the tick labels carry only what differs within a group (the
    basis; the source), and the shared prefix (env · run, or env · instance) is written
    once, vertically centred on its block, with a rule between blocks."""
    y, ys, texts = 0, [], []
    for i, (lab, n) in enumerate(groups):
        if i:
            ys.append(y)
        texts.append(ax.text(x, y + n / 2, lab, transform=ax.get_yaxis_transform(),
                             ha="right", va="center", fontsize=9, color=RULE))
        y += n
    block_rules(ax, ys, texts)

RUN_GROUP = lambda df: groups_of(df.assign(g=[f"{r.env} · {r.run}" for r in df.itertuples()]), "g")

In [ ]:
# [1b] Which discworld rows use the FRAME-SET Edit Index (2026-09-05). A frames-as-tokens
#      run emits a distribution over next frames, so its Edit Index is Othello's legal-set
#      construction on the two worlds' frames at the edit frame — the same axis as the
#      ray-zone index of the other discworld rows, not the same formula (both are step-0).
#      Such runs carry `ei_construction: "frame-set"` in scores.json and are marked † in
#      every table below; ND stays blank for them as for every discworld run.
#      Scans through `counted` from cell [1], so an excluded run cannot sneak a † in here.
FRAME_SET = set()
for sp in sorted((REPO / "runs").rglob("scores.json")):
    rel = sp.relative_to(REPO / "runs")
    if not counted(rel):
        continue
    if json.loads(sp.read_text()).get("ei_construction") == "frame-set":
        FRAME_SET.add(sp.parent.name)
for _df in (DF, DF_X, DF_ALL, PERDIM, PERDIM_ALL):
    if len(_df):
        _df["run"] = [f"{r} †" if r in FRAME_SET else r for r in _df["run"]]
print("† = frame-set Edit Index (frames-as-tokens model):", sorted(FRAME_SET) or "none yet")

In [ ]:
# [2] TABLE 1 — DECODABILITY. Read before Table 2 means anything.
#     Probe Skill at the best residual point; 1 = perfect, 0 = the trivial baseline.
#     All probes held out BY SEQUENCE in both environments. Rows: Othello first, then the
#     discworld runs (transformer, then recurrent; noisy instance before noiseless); each
#     run's two bases share the run label and are told apart by the basis alone.
fig, ax = plt.subplots(figsize=(4.6, 0.52 * len(DF) + 1.4))
heat(ax, DF[["skill_LIN", "skill_MLP"]].values, ["LIN", "MLP-128"], list(DF["basis"]),
     fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0,
     cbar_label="Probe Skill", title="Table 1 — decodability (best residual point)")
group_rows(ax, RUN_GROUP(DF))
for i, t in enumerate(DF["tripwire"].values):      # tripwire = MLP < linear somewhere
    if t:
        ax.add_patch(plt.Rectangle((0, i), 2, 1, fill=False, edgecolor="#c0392b", lw=3))
        ax.text(2.06, i + 0.5, f"⚠ {t}", color="#c0392b", va="center", fontsize=9)
plt.show()

In [ ]:
# [3] TABLES 1b / 1c — DISCWORLD DECODABILITY BY COMPONENT (full target, best point),
#     one table per probe family. The aggregate in Table 1 is variance-weighted ~1000:1
#     toward position; velocity is only visible here. (Othello's per-tile equivalent is
#     64 columns — it stays in each run's scores.json.) Same row order as Table 1.
for tag, fam in (("1b", "LIN"), ("1c", "MLP")):
    P = PERDIM[PERDIM["probe"] == fam].reset_index(drop=True)
    if not len(P):
        print(f"Table {tag}: no discworld runs with per-component decodability yet")
        continue
    fig, ax = plt.subplots(figsize=(8.6, 0.52 * len(P) + 1.5))
    heat(ax, P[list(COMPONENTS)].values, list(COMPONENTS), list(P["basis"]),
         fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0, cbar_label="Probe Skill",
         title=f"Table {tag} — discworld decodability by component, {fam}"
               + ("-128" if fam == "MLP" else ""))
    group_rows(ax, RUN_GROUP(P), x=-0.16)
    ax.axvline(4, color="#172239", lw=2)          # position | velocity
    n = len(P)
    ax.text(2, n + 0.62, "position", ha="center", fontsize=9.5, color="0.3")
    ax.text(6, n + 0.62, "velocity", ha="center", fontsize=9.5, color="0.3")
    plt.show()

In [ ]:
# [3b] TABLES 1d / 1e — DISCWORLD DECODABILITY BY COMPONENT, ABOVE THE RANDOM-INIT FLOOR.
#     Each cell is Table 1b (LIN) / 1c (MLP-128) MINUS the same component read from the
#     SAME architecture at random initialisation on the SAME instance and basis (the
#     binding floor of Table 3): what training added, per state variable, possibly
#     negative. The random-init per-component skills are read from the cached baseline
#     probes (runs/_baselines/<instance>/probes/*.pt — every fit keeps its per-dim R²), at
#     the random-init LIN probe's own best point, mirroring 1b/1c's rule for the trained
#     rows. Nothing is refitted here.
#     Architecture of a cached baseline probe: the recurrent model by its span (> 100), a
#     frames-as-tokens model by the `encoder` in its provenance (2026-09-05), else the
#     regression transformer — three floors that share an instance must not overwrite
#     one another.
import torch

BASELINES = REPO / "runs" / "_baselines"        # also used by Table 3 and Fig 1 below

def probe_arch(prov):
    if int(prov.get("span", 39)) > 100:
        return "recurrent_l"
    if prov.get("encoder"):
        return "transformer_l_tokens"
    return "transformer_l"

def row_arch(arch):
    """The trained row's arch, in the same vocabulary as probe_arch."""
    arch = str(arch)
    if arch.startswith("recurrent"):
        return "recurrent_l"
    return "transformer_l_tokens" if arch.endswith("_tokens") else "transformer_l"

def baseline_perdim(instance):
    """{(arch, family, basis): (point, per-component skill) at the random-init LIN best point}."""
    out, by = {}, {}
    for pt_ in sorted((BASELINES / instance / "probes").glob("probes_*.pt")):
        blob = torch.load(pt_, map_location="cpu", weights_only=False)
        prov = blob["provenance"]
        if prov.get("model", "none") == "none" or prov.get("target") != "full":
            continue                                  # observation probes: not this table
        by[(probe_arch(prov), prov["family"], prov["basis"])] = blob["probes"]   # {point: (probe, stats)}
    for (arch, fam, basis), probes in by.items():
        lin = by.get((arch, "linear", basis))
        if lin is None:
            continue
        bp = max(lin, key=lambda p: lin[p][1]["r2"])   # the random-init LIN best point
        out[(arch, fam, basis)] = (bp, np.array(probes[bp][1]["per_dim_r2"][:len(COMPONENTS)], float))
    return out

RAND = {inst: baseline_perdim(inst) for inst in INSTANCE_ORDER
        if (BASELINES / inst / "probes").exists() and inst.startswith("dw")}
ref_rows = []
for tag, fam, famkey in (("1d", "LIN", "linear"), ("1e", "MLP", "mlp")):
    P = PERDIM[PERDIM["probe"] == fam].reset_index(drop=True)
    if not len(P):
        continue
    delta = []
    for i, r in P.iterrows():
        bp, base = RAND.get(r["instance"], {}).get((row_arch(r["arch"]), famkey, r["basis"]),
                                                    (None, np.full(len(COMPONENTS), np.nan)))
        delta.append(P.loc[i, list(COMPONENTS)].values.astype(float) - base)
        ref_rows.append({"table": tag, "run": r["run"], "basis": r["basis"],
                         "random-init point": bp, **dict(zip(COMPONENTS, base))})
    D = np.array(delta, float)
    fig, ax = plt.subplots(figsize=(8.6, 0.52 * len(P) + 1.5))
    heat(ax, D, list(COMPONENTS), list(P["basis"]), fmt="+.3f", cmap="RdYlGn",
         vmin=-1.0, vmax=1.0, cbar_label="Probe Skill − random-init floor",
         title=f"Table {tag} — discworld decodability by component ABOVE the random-init "
               f"floor, {fam}{'-128' if fam == 'MLP' else ''}")
    group_rows(ax, RUN_GROUP(P), x=-0.16)
    ax.axvline(4, color="#172239", lw=2)
    n = len(P)
    ax.text(2, n + 0.62, "position", ha="center", fontsize=9.5, color="0.3")
    ax.text(6, n + 0.62, "velocity", ha="center", fontsize=9.5, color="0.3")
    plt.show()
print("random-init per-component skill that was subtracted (at the random-init LIN best point):")
display(pd.DataFrame(ref_rows).set_index(["table", "run", "basis"]).round(3))

In [ ]:
# [4] TABLE 2 — EDITABILITY. Two panels, never mixed in one colour scale:
#     (a) Edit Index, prefixed by each run's UNEDITED floor — where its −1 end sits.
#     (b) the guard: RMSE(edited, GT)/RMSE(unsteered, GT) at the edit step.
#         >1 = degraded, not steered. Colormap REVERSED so green = good in both panels.
#     Same row order and two-level labels as Table 1.
ei = DF[["unedited"] + [f"{e} EI" for e in EDITORS]].values
fid = DF[[f"{e} fid" for e in EDITORS]].values
fig, axes = plt.subplots(1, 2, figsize=(12.6, 0.52 * len(DF) + 2.0),
                         gridspec_kw=dict(width_ratios=[5, 4], wspace=0.35))
heat(axes[0], ei, ["unedited"] + list(EDITORS), list(DF["basis"]), fmt="+.3f", cmap="RdYlGn",
     vmin=-1.0, vmax=1.0, cbar_label="Edit Index",
     title="(a) did the edit land?   +1 = the edited world, −1 = the unedited one")
group_rows(axes[0], RUN_GROUP(DF), x=-0.22)
axes[0].axvline(1, color="#172239", lw=2)          # floor | editors
heat(axes[1], fid, list(EDITORS), [""] * len(DF), fmt=".2f", cmap="RdYlGn_r",
     norm=TwoSlopeNorm(vmin=0.0, vcenter=1.0, vmax=3.0), cbar_label="fidelity ratio",
     title="(b) did the world survive?   1.0 = doing nothing")
y = 0
for _, n in RUN_GROUP(DF)[:-1]:                    # the same block rules on panel (b)
    y += n
    axes[1].axhline(y, color="#172239", lw=1.6)
fig.suptitle("Table 2 — editability, every canonical editor", fontsize=12, y=1.0)
plt.show()

# TABLE 2b — the winning arm behind each cell, so a number can always be traced back to
# a configuration. On discworld each arm names the DIM SET that won it: "pos" = the edit
# drove the position read-outs only, "all" = it drove the whole state. Both are swept
# through the same full-state probe set and the better one is what Table 2 reports.
print("Table 2b — best arm per cell   (discworld: dims·point·α · Othello: point·α)")
display(DF.set_index(["env", "run", "basis"])[[f"{e} arm" for e in EDITORS]])

In [ ]:
# [4c] TABLE 2c — THE PROBE-TARGET SWEEP (2026-09-10). Every EXTRA probe target in one
#      table, so Tables 1-3 keep only the canonical bases: the categorical partitions of
#      discworld (the frustum product grids `grid-<nu>x<nd>`, the appearance family — the
#      observation-exact partition and its `-lat` coarsening / `-d<k>` refinements — and the
#      misaligned 30-cell controls `grid-6x5`, `grid-10x3`), the SNAPPED regression targets
#      (`pos@<partition>`: the same cells read as 4-D position regression) and Othello's
#      signed regression target `mine_signed`. Rows: per run, the run's canonical row first
#      as the REFERENCE (discworld: frustum regression; Othello: mine/theirs), then its extra
#      targets by CELL COUNT, coarse → fine — the count is the target's own on the run's
#      instance (Othello: 64 tiles). Panels as Tables 1 / 2: (a) Probe Skill, (b) Edit Index
#      with the unedited floor, (c) the fidelity guard. ND is blank on a discworld regression
#      target (ill-posed there), reported on categorical targets and on Othello.
import h5py
from pim.environments import layout
from pim.environments.discworld.grid_target import target_cells

def _sim_of(inst):
    with h5py.File(layout.edits_file("discworld", inst)) as f:
        return json.loads(f.attrs["config_json"])["dataset"]["sim"]

if not len(DF_X):
    print("Table 2c: no extra probe-target rows yet")
else:
    SIMS = {i: _sim_of(i) for i in set(DF_X[DF_X["env"] == "discworld"]["instance"])}
    REF_KEY = {"discworld": "frustum", "othello": "mine/theirs"}
    parts = []
    for run in DF_X["run"].drop_duplicates():                 # DF_X keeps Table 1's run order
        X = DF_X[DF_X["run"] == run].copy()
        env = X["env"].iloc[0]
        X["cells"] = [target_cells(b, SIMS[i]) if env == "discworld" else 64
                      for b, i in zip(X["basis"], X["instance"])]
        X = X.sort_values(["cells", "basis"])
        X["label"] = [f"{b} · {c} {'tiles' if env == 'othello' else 'cells'}"
                      for b, c in zip(X["basis"], X["cells"])]
        ref = DF[(DF["run"] == run) & (DF["basis"] == REF_KEY[env])].copy()
        ref["cells"], ref["label"] = np.nan, f"{REF_KEY[env]} · reference"
        parts += [ref, X]
    T2C = pd.concat(parts, ignore_index=True)
    ei = T2C[["unedited"] + [f"{e} EI" for e in EDITORS]].values
    fid = T2C[[f"{e} fid" for e in EDITORS]].values
    fig, axes = plt.subplots(1, 3, figsize=(17.5, 0.5 * len(T2C) + 2.2),
                             gridspec_kw=dict(width_ratios=[2.6, 5, 3.4], wspace=0.5))
    heat(axes[0], T2C[["skill_LIN", "skill_MLP"]].values, ["LIN", "MLP-128"], list(T2C["label"]),
         fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0, cbar_label="Probe Skill",
         title="(a) decodability — best residual point")
    group_rows(axes[0], RUN_GROUP(T2C), x=-1.05)      # clear of the longest row label
    heat(axes[1], ei, ["unedited"] + list(EDITORS), [""] * len(T2C), fmt="+.3f", cmap="RdYlGn",
         vmin=-1.0, vmax=1.0, cbar_label="Edit Index",
         title="(b) did the edit land?   +1 = the edited world, −1 = the unedited one")
    axes[1].axvline(1, color="#172239", lw=2)
    heat(axes[2], fid, list(EDITORS), [""] * len(T2C), fmt=".2f", cmap="RdYlGn_r",
         norm=TwoSlopeNorm(vmin=0.0, vcenter=1.0, vmax=3.0), cbar_label="fidelity ratio",
         title="(c) did the world survive?   1.0 = doing nothing")
    y = 0
    for _, n in RUN_GROUP(T2C)[:-1]:                          # the same block rules on (b), (c)
        y += n
        for ax in axes[1:]:
            ax.axhline(y, color="#172239", lw=1.6)
    fig.suptitle("Table 2c — the probe-target sweep: every extra probe target, coarse → fine, "
                 "under each run's canonical reference row", fontsize=12, y=1.0)
    plt.show()
    print("Table 2c — table view (cells = the target's partition on the run's instance; "
          "kind = what the probe reads; arms as in Table 2b)")
    display(T2C.set_index(["env", "run", "label"])[["kind", "cells", "skill_LIN", "skill_MLP", "unedited"]
                                                  + [f"{e} EI" for e in EDITORS] + [f"{e} fid" for e in EDITORS]
                                                  + [f"{e} arm" for e in EDITORS]].round(3))

In [ ]:
# [5] TABLES 3a / 3b / 3c… — DECODABILITY BASELINES. A separate table per probe target
#     (3a cartesian, 3b frustum — Othello's mine/theirs rows appear in both; then one per
#     extra discworld target, discworld only; then one per extra Othello target, Othello
#     only): the two floors every Probe Skill in Table 1 has to be read against.
#
#       observation   the SAME probes fitted to the causal INPUT history instead of the
#                     residual stream — how much a shallow read of the input already gives
#       random-init   the SAME architecture, seeded, never trained, probed identically —
#                     how much comes from training rather than from random features
#       trained       the model rows from Table 1, repeated so the gap is visible
#
#     ONE observation row per instance and LAYOUT. The observation probe reads the input
#     history, not a model, so it is one measurement per instance — the two architectures'
#     spans (39 and 10,000) both cover a 40-frame sequence — and it is quoted from the LARGE
#     corpus (master_eval b3: discworld 250k sequences / Othello 170k games, 50 epochs)
#     where that has run, because on the matched 30k corpus the wide-input probe (discworld
#     39x128 = 4,992 features vs the model's 512) memorises its train split — panel (b) is
#     that overfit check, in-sample minus held-out on the skill scale — and its floor is an
#     under-estimate. Two LAYOUTS (b4, 2026-09-06): "left" is the original (block j = frame
#     j), under which a LINEAR probe cannot express a current-frame lookup, so its LIN cell
#     understates a linear read of the input; "right" lays the history out relative to the
#     present (block 0 = now). Read the right-aligned LIN cell as the linear floor; the MLP
#     barely differs between layouts. Random-init is per architecture by construction: a
#     trained run sits under the random-init row of ITS OWN architecture (exact match — the
#     regression transformer and the frames-as-tokens transformer are different floors,
#     2026-09-05). Rows follow the Table 1 order: Othello, then the discworld instances;
#     transformer before recurrent. ⚠ The binding floor is the HIGHER of the two.
#     A discworld extra target's floors (2026-09-09) carry only the right-aligned
#     large-corpus observation row — the layout and corpus its probes were fitted on.
BASELINES = REPO / "runs" / "_baselines"
DF_T3 = pd.concat([DF, DF_X], ignore_index=True) if len(DF_X) else DF   # canonical + extra rows

def _cells(d, note):
    return {"skill_LIN": d["linear"]["skill"], "skill_MLP": d["mlp"]["skill"],
            "gap_LIN": d["linear"]["insample_gap"], "gap_MLP": d["mlp"]["insample_gap"],
            "note": note}

ARCH_LABEL = {"transformer_l": "L", "transformer_s": "S", "recurrent_l": "R",
              "transformer_l_tokens": "L (tokens)", "transformer_s_tokens": "S (tokens)"}

BASE = {}                                     # instance -> its baselines.json
for bp_ in BASELINES.glob("*/baselines.json"):
    b = json.loads(bp_.read_text())
    if "archs" not in b:
        raise RuntimeError(f"{bp_} predates the per-arch schema — rerun master_eval [5]")
    BASE[b["instance"]] = b

def _obs_rows(A0, env):
    """The observation-floor rows of one instance block: one per layout, the large corpus
    preferred (the matched corpus as the fallback, labelled as such)."""
    rows = []
    unit = "games" if env == "othello" else "seq"
    for layout, large_key, small_key in (("left", "observation_large", "observation"),
                                         ("right", "observation_right_large", "observation_right")):
        if A0.get(large_key, {}).get("mlp"):
            L = A0[large_key]
            rows.append((f"observation · {layout}-aligned · {L['mlp']['n_seq'] // 1000}k {unit}",
                         _cells(L, f"large corpus · {L['mlp']['epochs']} epochs · {layout}-aligned history")))
        elif A0.get(small_key, {}).get("mlp"):
            rows.append((f"observation · {layout}-aligned · matched", _cells(A0[small_key],
                         f"matched corpus (30k seq / 20k games) — under-estimate, see (b) · {layout}-aligned")))
    return rows

def table3_rows(dkey, okey):
    """Instance blocks in INSTANCE_ORDER; inside a block: the observation floors, then for
    each architecture (transformer first) its random-init floor and its trained run(s).
    `dkey` = the discworld block key (None = no discworld rows), `okey` = the Othello one."""
    out = []
    for inst in INSTANCE_ORDER:
        b = BASE.get(inst)
        if b is None:
            continue
        bkey = okey if b["env"] == "othello" else dkey
        if bkey is None:
            continue
        archs = sorted((a for a in b["archs"] if bkey in b["archs"][a]["bases"]),
                       key=lambda a: order_key(b["env"], a, inst))
        if not archs:
            continue
        block = _obs_rows(b["archs"][archs[0]]["bases"][bkey], b["env"])
        for a in archs:
            A = b["archs"][a]["bases"][bkey]
            block.append((f"random-init · {ARCH_LABEL.get(a, a)}", _cells(A["random_init"], a)))
            for r in DF_T3[(DF_T3["instance"] == inst) & (DF_T3["basis"] == bkey)].itertuples():
                if r.arch == a:
                    block.append((f"trained · {r.run}",
                                  {"skill_LIN": r.skill_LIN, "skill_MLP": r.skill_MLP,
                                   "gap_LIN": r.gap_LIN, "gap_MLP": r.gap_MLP,
                                   "note": r.arch}))
        for src, cells in block:
            out.append({"env": b["env"], "instance": inst, "basis": bkey, "source": src,
                        **cells})
    return pd.DataFrame(out)

if not BASE:
    print("no runs/_baselines/*/baselines.json yet — run master_eval cell [5]")
T3 = [("3a", "cartesian", "mine/theirs", "discworld cartesian basis (Othello mine/theirs in both)"),
      ("3b", "frustum", "mine/theirs", "discworld frustum basis (Othello mine/theirs in both)")]
T3 += [(f"3{chr(ord('c') + i)}", t, None, f"discworld {t} target ({KIND_X.get(t, '?')}; discworld only)")
       for i, t in enumerate(EXTRA_TARGETS)]
T3 += [(f"3{chr(ord('c') + len(EXTRA_TARGETS) + i)}", None, t, f"Othello {t} target (regression; Othello only)")
       for i, t in enumerate(EXTRA_TARGETS_OTH)]
for tag, dkey, okey, what in T3:
    B = table3_rows(dkey, okey)
    if not len(B):
        continue
    groups = groups_of(B.assign(g=[f"{r.env} · {r.instance}" for r in B.itertuples()]), "g")
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 0.5 * len(B) + 2.2),
                             gridspec_kw=dict(width_ratios=[1, 1], wspace=0.62))
    heat(axes[0], B[["skill_LIN", "skill_MLP"]].values, ["LIN", "MLP-128"],
         list(B["source"]), fmt="+.3f", cmap="Greens", vmin=0.0, vmax=1.0,
         cbar_label="Probe Skill", title="(a) Probe Skill — each floor against the trained model")
    group_rows(axes[0], groups, x=-0.72)      # clear of the longest 'trained · …' label
    heat(axes[1], B[["gap_LIN", "gap_MLP"]].values, ["LIN", "MLP-128"], [""] * len(B),
         fmt="+.3f", cmap="Oranges", vmin=0.0, vmax=0.5,
         cbar_label="in-sample − held-out",
         title="(b) overfit check — big = the probe memorised its train split")
    y = 0
    for _, n in groups[:-1]:                   # the same block rules on panel (b)
        y += n
        axes[1].axhline(y, color="#172239", lw=1.6)
    fig.suptitle(f"Table {tag} — decodability baselines · {what}", fontsize=12, y=1.0)
    plt.show()
    display(B.set_index(["env", "instance", "source"])[["note"]])

In [ ]:
# [6] FIG 1 — THE TRAINING CURVE: decodability, editability and the fidelity guard as a
#     function of TRAINING STEP, on each canonical run's own log-spaced checkpoints
#     (runs/training_curve/<run>_s<step>/, scored by the unchanged master_eval).
#     The question: does editability EMERGE after decodability has saturated (Othello),
#     and does discworld's ever move? Small multiples — one row per run, one quantity per
#     panel, never a second axis. Discworld rows use the canonical frustum basis.
#     Reference lines: the binding (random-init) floors from Table 3 in (a), each
#     checkpoint's own unedited index in (b), and 1.0 = doing nothing in (c) — every one
#     of them named in the legend. Colours follow the ENTITY across panels (Okabe-Ito,
#     validated 2026-09-01: adjacent CVD ΔE 11.0, normal-vision 21.1); marker shapes
#     differ per series, lines are end-labelled where they separate (converging lines
#     fall back to the legend rather than stacked labels), and the table underneath is
#     the relief channel for the sub-3:1 orange.
#
#     WHICH ARM (b)/(c) plot — ONE FIXED ARM PER EDITOR: the arm that wins at the LAST
#     checkpoint (Table 2's max-EI rule applied once, at the end), then read off at every
#     earlier checkpoint from the same sweep. Applying the argmax at every checkpoint is a
#     max over ~300 arms that can switch regime between neighbours — on L-dw-20m at 16k it
#     picks a last-layer α=175 write at fidelity 4.0 over the α=60 write it picks from 64k
#     on, an artefact of the selection rather than of training — so each checkpoint's own
#     argmax is drawn only as a HOLLOW marker where it differs from the tracked arm.
#     Nothing is dropped: both values come from the checkpoint's own scores.json, and
#     Table 2 is untouched (it still reports each run's max-EI arm).
import re

from matplotlib.lines import Line2D

from pim.figures.theme import PALETTE

CURVE = DF_ALL[DF_ALL["topic"] == "training_curve"].copy()
if not len(CURVE):
    print("no runs/training_curve/*/scores.json yet — launch "
          "experiments/training_curve/drivers/training_curve.sh")
else:
    CURVE["step"] = CURVE["run"].str.extract(r"_s(\d+)$")[0].astype(int)
    CURVE["source"] = CURVE["run"].str.replace(r"_s\d+$", "", regex=True)
    CURVE = CURVE[(CURVE["env"] != "discworld") | (CURVE["basis"] == "frustum")]
    CURVE = CURVE.sort_values(["env", "source", "step"]).reset_index(drop=True)

    def scored(run, env, basis):
        """(arms, best, edit-index key) of one checkpoint, from its own scores.json."""
        s = json.loads((REPO / "runs" / "training_curve" / run / "scores.json").read_text())
        if env == "discworld":
            T = s["bases"][basis]
            return T["arms"], T["best"], "edit_index"
        return s["arms"], s["best"], "edit_index_union"

    key_of = lambda a: (a["editor"], a["point"], a["alpha"], a.get("dims"))
    arm_str = lambda a: (f"{a['dims']}·" if "dims" in a else "") + f"pt{a['point']}·α{a['alpha']:g}"

    for ed in EDITORS:
        CURVE[f"{ed} EI·tracked"], CURVE[f"{ed} fid·tracked"] = np.nan, np.nan
        CURVE[f"{ed} tracked arm"] = "—"
    for src in dict.fromkeys(CURVE["source"]):
        d = CURVE[CURVE["source"] == src]
        env, basis = d.iloc[0]["env"], d.iloc[0]["basis"]
        _, final, _ = scored(d.iloc[-1]["run"], env, basis)
        for idx, r in d.iterrows():
            arms, _, k = scored(r["run"], env, basis)
            by_key = {key_of(a): a for a in arms}
            for ed in EDITORS:
                if (ed == "ND" and env == "discworld") or not final.get(ed):
                    continue                    # n/a on discworld, as in Table 2
                a = by_key.get(key_of(final[ed]))
                if a is None:
                    continue
                CURVE.loc[idx, f"{ed} EI·tracked"] = a[k]
                CURVE.loc[idx, f"{ed} fid·tracked"] = a["fidelity_ratio"]
                CURVE.loc[idx, f"{ed} tracked arm"] = arm_str(final[ed])

    hexc = lambda i: "#%02x%02x%02x" % tuple(int(round(v * 255)) for v in PALETTE[i])
    ENT = {"LIN": (hexc(0), "o"), "MLP": (hexc(1), "s"),           # decodability
           "PI": (hexc(3), "o"), "ND": (hexc(2), "s"), "GS": (hexc(4), "^")}  # editors
    INK2, REF, GRID = "#52514e", "#898781", "#e1e0d9"
    kfmt = lambda s: f"{s // 1000}k" if s >= 1000 else str(s)
    HOLLOW = Line2D([], [], ls="none", marker="o", mfc="white", mec=INK2, mew=1.4,
                    label="hollow: that checkpoint's argmax")

    def floors(instance, arch, basis):
        """The binding floors for panel (a): random-init LIN / MLP skill from Table 3."""
        p = BASELINES / instance / "baselines.json"
        if not p.exists():
            return None
        A = json.loads(p.read_text()).get("archs", {}).get(arch)
        if not A or basis not in A["bases"]:
            return None
        r = A["bases"][basis]["random_init"]
        return r["linear"]["skill"], r["mlp"]["skill"]

    def line(ax, x, y, name, ends, label=None):
        c, m = ENT[name]
        if y.isna().all():
            return
        ax.plot(x, y, color=c, lw=2, marker=m, ms=6, mec="white", mew=1.2,
                solid_capstyle="round", label=label or name)
        ends.append((name, float(x.iloc[-1]), float(y.iloc[-1])))

    def hollow(ax, x, y, name, mask):
        """Each checkpoint's own argmax arm, only where it is NOT the tracked arm."""
        c, m = ENT[name]
        if mask.any():
            ax.plot(x[mask], y[mask], ls="none", marker=m, ms=7, mfc="white", mec=c,
                    mew=1.6, label="_nolegend_")
        return bool(mask.any())

    def end_labels(ax, ends, frac=0.06):
        """Direct-label a line's end only where it is SEPARATED from every other end by
        more than `frac` of the axis range; converging lines keep the legend instead."""
        lo, hi = ax.get_ylim()
        for name, x, y in ends:
            if all(abs(y - y2) > frac * (hi - lo) for n2, _, y2 in ends if n2 != name):
                ax.annotate(name, (x, y), xytext=(6, 0), textcoords="offset points",
                            va="center", fontsize=8, color=INK2)

    def dress(ax, steps, title, ylabel, extra=(), loc="best"):
        ax.set_xscale("log")
        ax.set_xticks(steps)
        ax.set_xticklabels([kfmt(s) for s in steps], fontsize=8, rotation=35, ha="right")
        ax.minorticks_off()
        ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
        for sp_ in ax.spines.values():
            sp_.set_edgecolor("#c3c2b7")
        ax.set_title(title, fontsize=10, loc="left", pad=6)
        ax.set_xlabel("training step", fontsize=9, color=INK2)
        ax.set_ylabel(ylabel, fontsize=9, color=INK2)
        ax.tick_params(labelsize=8, colors=INK2)
        h, _ = ax.get_legend_handles_labels()
        ax.legend(handles=h + list(extra), fontsize=7.5, frameon=False, handlelength=2.4,
                  loc=loc)

    sources = list(dict.fromkeys(CURVE["source"]))
    fig, axes = plt.subplots(len(sources), 3, figsize=(15, 4.1 * len(sources)), squeeze=False,
                             gridspec_kw=dict(wspace=0.32, hspace=0.6))
    for i, src in enumerate(sources):
        d = CURVE[CURVE["source"] == src]
        x, steps = d["step"], list(d["step"])
        env, inst, arch, basis = d.iloc[0][["env", "instance", "arch", "basis"]]
        a, b_, c = axes[i]
        # (a) decodability
        fl = floors(inst, arch, basis)
        if fl:
            a.axhline(fl[0], color=REF, lw=1.2, label="random-init floor · LIN")
            a.axhline(fl[1], color=REF, lw=1.2, ls=":", label="random-init floor · MLP")
        ends = []
        line(a, x, d["skill_LIN"], "LIN", ends); line(a, x, d["skill_MLP"], "MLP", ends)
        a.set_ylim(0, 1.02); end_labels(a, ends)
        dress(a, steps, f"(a) {src} · {basis} — Probe Skill, best point", "Probe Skill")
        # (b) editability — the tracked arm; hollow = that checkpoint's own argmax
        b_.plot(x, d["unedited"], color=REF, lw=1.2, label="unedited (this checkpoint)")
        ends, any_hollow = [], False
        for ed in EDITORS:
            arm = d.iloc[-1][f"{ed} tracked arm"]
            differs = (d[f"{ed} arm"] != arm) & d[f"{ed} EI"].notna()
            line(b_, x, d[f"{ed} EI·tracked"], ed, ends, label=f"{ed} · {arm}")
            any_hollow |= hollow(b_, x, d[f"{ed} EI"], ed, differs)
        b_.set_ylim(-1, 1); b_.axhline(0, color="#c3c2b7", lw=0.8); end_labels(b_, ends)
        dress(b_, steps, f"(b) {src} — Edit Index, tracked arm", "Edit Index",
              extra=[HOLLOW] if any_hollow else [])
        # (c) the guard, same arms — legend top-left with headroom above the highest
        # value, so it can never cover a hollow marker (the 16k outlier on discworld)
        c.axhline(1.0, color=REF, lw=1.2, label="1.0 = doing nothing")
        ends, any_hollow = [], False
        for ed in EDITORS:
            arm = d.iloc[-1][f"{ed} tracked arm"]
            differs = (d[f"{ed} arm"] != arm) & d[f"{ed} fid"].notna()
            line(c, x, d[f"{ed} fid·tracked"], ed, ends, label=f"{ed} · {arm}")
            any_hollow |= hollow(c, x, d[f"{ed} fid"], ed, differs)
        top = np.nanmax(d[[f"{e} fid·tracked" for e in EDITORS]
                          + [f"{e} fid" for e in EDITORS]].values)
        c.set_ylim(0, max(3.0, min(float(top) * 1.4, 12.0))); end_labels(c, ends)
        dress(c, steps, f"(c) {src} — fidelity ratio at the edit step, tracked arm",
              "fidelity ratio", extra=[HOLLOW] if any_hollow else [], loc="upper left")
    fig.suptitle("Fig 1 — training curve: decodability, editability and the fidelity guard "
                 "vs training step (each run's own checkpoints)\n"
                 "(b)/(c): the arm that wins at the final checkpoint, read off at every "
                 "checkpoint; hollow markers = that checkpoint's own argmax where it differs",
                 fontsize=11.5, y=1.02)
    plt.show()

    cols = ["skill_LIN", "skill_MLP", "unedited"] \
           + [f"{e} EI·tracked" for e in EDITORS] + [f"{e} fid·tracked" for e in EDITORS] \
           + [f"{e} EI" for e in EDITORS] + [f"{e} fid" for e in EDITORS] + ["val"]
    print("Fig 1 — table view  (·tracked = the final checkpoint's arm; plain = that "
          "checkpoint's own argmax, Table 2's rule)")
    display(CURVE.set_index(["env", "source", "step"])[cols].round(4))
    display(CURVE.groupby(["env", "source"])[[f"{e} tracked arm" for e in EDITORS]].last())

In [ ]:
# [7] FIG 2 — THE PROBE-CAPACITY SWEEP: Probe Skill against one-hidden-layer width, one
#     residual point per environment, three sources (trained / random-init / observation),
#     on ~5x the canonical probe rows (discworld 250k sequences, Othello 170k games) so the
#     wide probes are limited by width, not by memorisation. The question: does Othello's
#     random reservoir also catch the trained model as the probe widens — just later than
#     discworld's — or never at this data? Hollow markers = the canonical corpus (30k/20k)
#     values at the same point, so the data effect at LIN and 128 is visible too.
#     Panel (b) is the overfit check; a rising gap is where a line stops being trustworthy.
#
#     LIVE: experiments/probe_capacity/scripts/probe_capacity.py rewrites its scores JSON
#     after EVERY fit (atomically), so re-running this cell while the sweep runs shows the
#     figure filling in. Missing cells break the line rather than being interpolated.
#     Drawing is pim.figures.probe_capacity.capacity_figure — the script's own PNG
#     (experiments/probe_capacity/outputs/probe_capacity.png) is the same function.
from pim.figures.probe_capacity import capacity_figure

CAP = REPO / "experiments" / "probe_capacity" / "scores"
cap_files = [CAP / f"probe_capacity_{e}.json" for e in ("discworld", "othello")]
if not any(p.exists() for p in cap_files):
    print("no experiments/probe_capacity/scores/*.json yet — launch "
          "experiments/probe_capacity/drivers/probe_capacity.sh")
else:
    fig = capacity_figure(cap_files)
    plt.show()
    print("Fig 2 — table view (skill / in-sample gap / params per cell)")
    rows_ = []
    for p in cap_files:
        if not p.exists():
            continue
        S = json.loads(p.read_text())
        for src, cells in S["cells"].items():
            for w, c in cells.items():
                rows_.append({"env": S["env"], "source": src, "width": w,
                              "skill": c["skill"], "gap": c["insample_gap"],
                              "params": c["params"], "min": c.get("minutes")})
    CAPDF = pd.DataFrame(rows_)
    display(CAPDF.set_index(["env", "source", "width"]).round(4))

In [ ]:
# [8] FIG 3 — THE TARGET-RESOLUTION SWEEP on dw-8ray (2026-09-09 → 10): decodability and
#     editability of the two 8-ray models as a function of the CATEGORICAL probe target's
#     resolution (number of cells, log axis). Two families: the APPEARANCE partition (cells
#     = the runs of rays a disc lights — the observation-exact partition, 30 cells — and its
#     coarsening `-lat` / refinements `-d2`, `-d3`), drawn as filled circles, and the frustum
#     PRODUCT grids (`grid-<nu>x<nd>`), drawn as hollow squares — same cell count does not
#     mean same alignment with the frame, which is the point. Reference lines: the run's
#     REGRESSION target (frustum basis) per editor (dashed, "what the continuous read-out
#     gives on the same residual stream") and its unedited floor (grey). One quantity per
#     panel, one axis; colours follow the entity as in Fig 1; a table view underneath.
from pim.figures.theme import PALETTE as _PAL

_hex = lambda i: "#%02x%02x%02x" % tuple(int(round(v * 255)) for v in _PAL[i])
ENT3 = {"LIN": _hex(0), "MLP": _hex(1), "PI": _hex(3), "ND": _hex(2), "GS": _hex(4)}
INK3, REF3, GRID3 = "#52514e", "#898781", "#e1e0d9"
SWEEP_RUNS = [r for r in ("L-dw-8ray-20m", "L-dw-8ray-tok-20m †", "L-dw-8ray-tok-20m") if r in set(DF_X["run"])]
S3 = DF_X[(DF_X["instance"] == "dw-8ray") & (DF_X["run"].isin(SWEEP_RUNS)) & (DF_X["kind"] != "regression")].copy()
if not len(S3):
    print("Fig 3: no categorical-target rows on dw-8ray yet")
else:
    # the cell count of every categorical target on this instance, from the target itself
    from pim.environments.discworld.grid_target import categorical_target
    import h5py
    from pim.environments import layout
    with h5py.File(layout.edits_file("discworld", "dw-8ray")) as _f:      # edits/v1 (layout v2)
        _sim = json.loads(_f.attrs["config_json"])["dataset"]["sim"]
    S3["cells"] = [categorical_target(b).n_cells(_sim) for b in S3["basis"]]
    S3["family"] = ["appearance" if b.startswith("appearance") else "grid" for b in S3["basis"]]
    S3 = S3.sort_values("cells")
    fig, axes = plt.subplots(len(SWEEP_RUNS), 2, figsize=(12.5, 4.0 * len(SWEEP_RUNS)), squeeze=False,
                             gridspec_kw=dict(wspace=0.28, hspace=0.5))
    for i, run in enumerate(SWEEP_RUNS):
        d = S3[S3["run"] == run]
        reg = DF[(DF["run"] == run) & (DF["basis"] == "frustum")].iloc[0]
        a, b_ = axes[i]
        for fam_, mk in (("appearance", "o"), ("grid", "s")):
            dd = d[d["family"] == fam_]
            for col, name in (("skill_LIN", "LIN"), ("skill_MLP", "MLP")):
                a.plot(dd["cells"], dd[col], ls="none", marker=mk, ms=8 if mk == "o" else 7,
                       mfc=ENT3[name] if mk == "o" else "white", mec=ENT3[name], mew=1.6,
                       label=f"{name} · {fam_}")
            for ed in EDITORS:
                b_.plot(dd["cells"], dd[f"{ed} EI"], ls="none", marker=mk, ms=8 if mk == "o" else 7,
                        mfc=ENT3[ed] if mk == "o" else "white", mec=ENT3[ed], mew=1.6,
                        label=f"{ed} · {fam_}")
        # the appearance family as a line (it is one ordered family: lat < exact < d2 < d3)
        dl = d[d["family"] == "appearance"].sort_values("cells")
        for col, name in (("skill_LIN", "LIN"), ("skill_MLP", "MLP")):
            a.plot(dl["cells"], dl[col], color=ENT3[name], lw=1.4, alpha=0.6, label="_nolegend_")
        for ed in EDITORS:
            b_.plot(dl["cells"], dl[f"{ed} EI"], color=ENT3[ed], lw=1.4, alpha=0.6, label="_nolegend_")
        # reference: the regression target on the same residual stream
        for col, name in (("skill_LIN", "LIN"), ("skill_MLP", "MLP")):
            a.axhline(reg[col], color=ENT3[name], lw=1.1, ls="--", alpha=0.8, label=f"{name} · regression (frustum)")
        for ed in ("PI", "GS"):                       # ND is not reported on the regression target
            b_.axhline(reg[f"{ed} EI"], color=ENT3[ed], lw=1.1, ls="--", alpha=0.8, label=f"{ed} · regression (frustum)")
        b_.axhline(reg["unedited"], color=REF3, lw=1.2, label="unedited floor")
        b_.axhline(0, color="#c3c2b7", lw=0.8)
        for _, r in d.iterrows():                    # direct labels: the target names
            a.annotate(r["basis"], (r["cells"], r["skill_MLP"]), xytext=(0, 7), textcoords="offset points",
                       ha="center", fontsize=7, color=INK3)
        for ax, title, ylab, ylim in ((a, f"(a) {run} — Probe Skill, best point", "Probe Skill", (0, 1.02)),
                                      (b_, f"(b) {run} — Edit Index, best arm per editor", "Edit Index", (-1, 1))):
            ax.set_xscale("log"); ax.set_ylim(*ylim)
            ax.set_xticks(sorted(set(d["cells"]))); ax.set_xticklabels([str(c) for c in sorted(set(d["cells"]))], fontsize=8)
            ax.minorticks_off(); ax.grid(True, color=GRID3, lw=0.8); ax.set_axisbelow(True)
            for sp_ in ax.spines.values():
                sp_.set_edgecolor("#c3c2b7")
            ax.set_title(title, fontsize=10, loc="left", pad=6)
            ax.set_xlabel("cells in the categorical target (log)", fontsize=9, color=INK3)
            ax.set_ylabel(ylab, fontsize=9, color=INK3)
            ax.tick_params(labelsize=8, colors=INK3)
            ax.legend(fontsize=7, frameon=False, ncol=2, loc="lower left" if ax is b_ else "lower right")
    fig.suptitle("Fig 3 — target-resolution sweep on dw-8ray: the observation-exact partition (appearance, 30 cells) "
                 "edits best;\ncoarser and finer targets and product grids of any size fall off — "
                 "filled = appearance family, hollow = product grids, dashed = the regression target",
                 fontsize=11, y=1.02)
    plt.show()
    print("Fig 3 — table view")
    display(S3.set_index(["run", "basis"])[["cells", "family", "skill_LIN", "skill_MLP", "unedited"]
                                          + [f"{e} EI" for e in EDITORS] + [f"{e} fid" for e in EDITORS]].round(3))